In [8]:
from getpass import getpass
from sqlalchemy import create_engine, URL

password = getpass("Enter PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="retailrocket_analytics"
)

engine = create_engine(connection_url)

In [9]:
from pathlib import Path

# Locate project directory
candidates = [Path.cwd(), Path.cwd().parent]

project_dir = next(
    (p for p in candidates if (p / "sql").exists() and (p / "Dashboard").exists()),
    None
)

if project_dir is None:
    raise FileNotFoundError("Could not locate project directory")

output_file = (
    project_dir / "Dashboard" / "Dashboard Data" / "product_performance.csv"
)

if output_file.exists():
    with open(output_file, "r", encoding="utf-8-sig") as file:
        total_rows = sum(1 for _ in file) - 1

    print("CSV exists:", True)
    print("Data rows:", total_rows)
    print("Expected rows:", 234838)
    print("Complete:", total_rows == 234838)
else:
    print("CSV file not found.")

CSV exists: True
Data rows: 234838
Expected rows: 234838
Complete: True


In [10]:
import csv
from pathlib import Path
from sqlalchemy import text


sql_file = project_dir / "sql" / "05_product_performance.sql"
output_file = project_dir / "Dashboard" / "Dashboard Data" / "product_performance.csv"
temp_file = output_file.with_name("product_performance_temp.csv")


query = sql_file.read_text(encoding="utf-8")

output_file.parent.mkdir(parents=True, exist_ok=True)

total_rows = 0

with engine.connect().execution_options(stream_results=True) as conn:
    result = conn.execute(text(query))
    result = result.yield_per(10000)

    try:
        with open(temp_file, "w", newline="", encoding="utf-8-sig") as file:
            writer = csv.writer(file)
            writer.writerow(result.keys())

            while True:
                batch = result.fetchmany(10000)

                if not batch:
                    break

                writer.writerows(batch)
                total_rows += len(batch)
                print(f"Exported: {total_rows:,}")

    finally:
        result.close()

if total_rows == 234838:
    temp_file.replace(output_file)
    print("\nExport successful:", output_file)
else:
    print(f"\nExport incomplete: {total_rows:,} / 234,838 rows")
    print("Temporary file retained for diagnosis.")

Exported: 10,000
Exported: 20,000
Exported: 30,000
Exported: 40,000
Exported: 50,000
Exported: 60,000
Exported: 70,000
Exported: 80,000
Exported: 90,000
Exported: 100,000
Exported: 110,000
Exported: 120,000
Exported: 130,000
Exported: 140,000
Exported: 150,000
Exported: 160,000
Exported: 170,000
Exported: 180,000
Exported: 190,000
Exported: 200,000
Exported: 210,000
Exported: 220,000
Exported: 230,000
Exported: 234,838

Export successful: d:\PROJECT 1 - PRODUCT ANALYSIS\project1_starter\Dashboard\Dashboard Data\product_performance.csv


In [11]:
with open(output_file, "r", encoding="utf-8-sig") as file:
    rows = sum(1 for _ in file) - 1

print("Exported CSV rows:", rows)
print("Export complete:", rows == 234838)
print("File location:", output_file)

Exported CSV rows: 234838
Export complete: True
File location: d:\PROJECT 1 - PRODUCT ANALYSIS\project1_starter\Dashboard\Dashboard Data\product_performance.csv
